# 21 · Results summary and deliverables

Everything below is loaded from artifacts written by Notebooks 01–20 (nothing is recomputed),
in the order of the deliverables list of the project plan.

In [ ]:
%run common/00_config.ipynb
from IPython.display import Image, Markdown, display


def show(name, width=1000):
    display(Image(filename=str(P["figures"] / f"{name}.png"), width=width))


def table(name, **kw):
    return pd.read_csv(P["tables"] / name, **kw)

## 1. Dataset description and exact version statement

In [ ]:
ds = load_json(P["results"] / "dataset_summary.json")
dl = load_json(P["results"] / "download_report.json")
display(Markdown(
    f"**CirCor DigiScope Phonocardiogram Dataset, version {dl['version']}** — {dl['files_verified']:,} files "
    f"({dl['total_size_MB']:.0f} MB) verified against `SHA256SUMS.txt`. Download command: `{dl['download_command']}`.\n\n"
    f"{ds['recordings']:,} recordings (4 kHz, 16-bit) from {ds['patients']} patients; "
    f"patient murmur: {ds['patient_murmur']}; recording-level murmur: {ds['recording_murmur']}; "
    f"locations: {ds['locations']}; mean duration {ds['duration_s']['mean']:.1f} s."))
table("split_summary.csv", index_col=0)

## 2. Preprocessing flowchart

```
WAV 4 kHz ─► QC (readable, silent, <4 s removed; clipping & low SNR flagged)
          ─► trim to TSV-annotated span ─► polyphase resample to 2 kHz
          ─► 4th-order Butterworth band-pass 20–800 Hz (zero-phase) ─► Schmidt spike removal
          ─► median/IQR normalisation ─► clip |x| at the 99.9th percentile
```

In [ ]:
show("03_preprocessing_chain")
table("qc_summary.csv", index_col=0).T

## 3. Signal-processing findings (24AIM301)

In [ ]:
display(table("04_filter_comparison.csv"), table("04_heart_rate_validation.csv"), table("04_image_feature_separability.csv"))
show("04_psd_by_class"); show("04_tf_image_processing")

## 4. RL action space and reward definition

In [ ]:
A = CFG["actions"]
display(pd.DataFrame({"component": ["wavelet", "level", "band (Hz)", "window (s)", "fusion"],
                      "options": [", ".join(A["wavelet"]), ", ".join(map(str, A["level"])), ", ".join(f"{a}-{b}" for a, b in A["band"]),
                                  ", ".join(map(str, A["window_s"])), ", ".join(A["fusion"])],
                      "count": [len(A["wavelet"]), len(A["level"]), len(A["band"]), len(A["window_s"]), len(A["fusion"])]}))
w = CFG["reward"]["weights"]
display(Markdown(f"**Reward** r_i(c) = {w['balanced']} · Δbalanced + {w['sensitivity']} · Δsensitivity + {w['specificity']} · Δspecificity "
                 f"+ {w['robustness']} · Δrobustness(10 dB pink noise) − {w['complexity']} · complexity, per recording, out-of-fold, "
                 f"relative to the baseline configuration db6 / L5 / 20–800 Hz / 2 s / TSW."))

## 5. RL results (policy level, proxy classifier, validation split)

In [ ]:
display(table("08_context_count_selection.csv"))
display(table("09_wavelet_selection_comparison.csv").pivot(index="method", columns="split", values=["macro_f1", "auroc"]).round(3))
rl = pd.concat([table("11_mc_td_results.csv"), table("12_planning_results.csv"), table("13_ppo_results.csv").rename(columns={"val_reward": "val_mean_reward", "macro_f1": "val_macro_f1", "auroc": "val_auroc"})])
display(rl.groupby("method", sort=False)[["fraction_of_DP_value", "val_mean_reward", "val_macro_f1", "val_auroc"]].mean().round(4))
show("11_td_mc_learning_curves"); show("12_planning_learning_curves")

## 6. Main results (test split, recording level, mean ± std over 3 seeds)

In [ ]:
display(table("18_main_results_recording_level.csv", index_col=0))
display(table("18_main_results_patient_level.csv", index_col=0))
display(load_json(P["results"] / "bootstrap_auroc_difference.json"))
show("18_roc_pr")

## 7. Ablations

In [ ]:
display(table("14_rl_ablations_proxy.csv", index_col=0).round(3))
display(table("17_deep_ablations.csv", index_col=0).round(3))

## 8. Robustness analysis

In [ ]:
rob = table("19_noise_robustness.csv")
display(rob.pivot_table(index=["noise", "snr_db"], columns="model", values="macro_f1", sort=False).round(3))
show("19_noise_robustness"); show("19_compression")

## 9. Selected actions and explainability

In [ ]:
show("20_selected_actions"); show("20_branch_attention"); show("20_attention_by_phase"); show("20_representative_regions")